# Single-Cell Transcriptomics Workshop Part 2: 
## Clustering, Cell Type Annotation, and Downstream Analysis with Scanpy

In Part 1, we preprocessed and filtered our dataset based on quality control (QC) metrics, and so we are ready to continue using Scanpy to analyze a scRNA-Seq dataset. Remember the details of our dataset patient cohort:  

For all samples, single-cell RNA sequencing ([10x Genomics](https://www.10xgenomics.com/what-is-single-cell-rna-seq)) was performed using peripheral blood mononuclear cells (PBMCs). Samples were collected from:
* Healthy donors (**n = 4**)
* Patients with COVID-19 of varying clinical severity, including severe, mild, and asymptomatic (**n = 8**)

*[Lee et al., 2020](https://doi.org/10.1126/sciimmunol.abd1554)*

---

Be mindful that this notebook will guide you through each step, but **you will need to complete code snippets** to get familiar with the structure and syntax of the analysis. 

Your best friends will be the [Scanpy documentation](https://scanpy.readthedocs.io/en/stable/index.html) and the [single-cell best practices guide](https://www.sc-best-practices.org/preamble.html) that this workshop is based on. Both of these resources will have more detailed information if at any step you are unsure or have more questions. 

This workshop is aimed to be a **high-level overview** to help you get familiar with the entire scRNA-Seq analysis pipeline. By no way are we going in depth at any of the many complex steps in the analysis: if you are curious to read more about any step along the way, we recommend the single cell best practices guide which references literature for future reading. 

Have fun!

In [ ]:
# if running in GOOGLE COLAB: run this cell once (ignore any errors!)
!pip install --quiet scanpy matplotlib seaborn gdown celltypist pydeseq2 igraph leidenalg

In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import scanpy as sc
import matplotlib.pyplot as plt
import celltypist
from celltypist import models
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import scipy.sparse as sp

# silence warnings
sc.settings.verbosity = 0

In [ ]:
# running in GOOGLE COLAB, load scRNA-Seq data from online here
import gdown
url = "https://drive.google.com/file/d/1p3hMKbg7S04Ufzpmc8v2WKDY5z3SduxR/view?usp=sharing"
gdown.download(url, "adata_preprocessed.h5ad", fuzzy=True, quiet=True)
adata = sc.read_h5ad("adata_preprocessed.h5ad")
adata

In [ ]:
# running LOCALLY: read in the already preprocessed data
adata = sc.read_h5ad("data/adata_preprocessed.h5ad")

In [ ]:
# plot UMAPs of data to remember what it looks like
sc.pl.umap(adata, color=["total_counts", "batch", "doublet_score"], ncols=3)

## Clustering

Preprocessing and visualization enabled us to describe our scRNA-seq dataset and reduce its dimensionality. Up to this point, we visualized cells to understand the underlying properties of our dataset. However, they are still rather abstractly defined. The next step in single-cell analysis is the **identification of cellular structure in the dataset**.

We describe cellular structure by finding cell identities that relate to known cell states or cell cycle stages. This process is usually called **cell identity annotation**. For this purpose, we structure cells into clusters to infer the identity of similar cells. 

**Clustering** itself is a common unsupervised machine learning problem. We can derive clusters by minimizing the "distance" (or similarity) of cells within a cluster in the reduced expression space. In this case, the expression space determines the gene expression similarity of cells with respect to a dimensionality-reduced representation. This lower-dimensional representation is, for example, determined with a principal-component analysis, and the similarity scoring is then based on Euclidean distances.

---

The **Leiden algorithm** is a community detection method that identifies natural groups within network data. In single-cell data, it finds clusters of highly connected cells. 

First, the algorithm computes a k-nearest neighbors (KNN) graph obtained from the principal component (PC) reduced expression space. 

Then, the algorithm starts with a basis clustering, where the starting point is a partition (cluster assignments across the whole graph) in which each cell functions as its own group. As a next step, the algorithm creates "better" partitions by moving individual cells from one group to another: this refinement is based on a distance measure to minimize the distance of cells within the same group (i.e. cluster cells together that are very "similar"). 

Then, the individual cells that are grouped together are aggregated into a higher level, coarse aggregate network. Subsequently, the algorithm moves individual nodes (groups of similar cells) in the aggregate network, until refinement no longer changes the partition (an optimum is reached). All steps are repeated until the final clustering is created and partitions no longer change.

<img src="https://www.sc-best-practices.org/_images/clustering.jpeg">

In [ ]:
# run default Leiden clustering and visualize the clusters on the UMAP plot

sc.tl.leiden(
    adata, key_added="leiden_res1", resolution=1.0, flavor="igraph", n_iterations=2
)
sc.pl.umap(adata, color="leiden_res1", legend_loc="on data")

## Exercise: Leiden clustering resolution

The Leiden algorithm has a **resolution** parameter that allows for determining the scale of the partition cluster and therefore the coarseness of the clustering. 

**A higher resolution parameter leads to more clusters.**

The following code tests different levels of Leiden resolutions. Run the code and check your understanding of the effect of the resolution parameter on the number of clusters identified. Remember that each cluster represents a group of cells. What biological group are we mathematically identifying here? What happens, biologically, when we make our clustering more "coarse" or "fine"? If you'd like, you can extend the code below to test other resolution values beyond those shown below. 

In [ ]:
sc.tl.leiden(
    adata, key_added="leiden_res0_25", resolution=0.25, flavor="igraph", n_iterations=2
)
sc.tl.leiden(
    adata, key_added="leiden_res0_5", resolution=0.5, flavor="igraph", n_iterations=2
)
sc.tl.leiden(
    adata, key_added="leiden_res1_5", resolution=1.5, flavor="igraph", n_iterations=2
)

sc.pl.umap(
    adata,
    color=["leiden_res0_25", "leiden_res0_5", "leiden_res1", "leiden_res1_5"],
    ncols=2,
    legend_loc="on data",
)

## Manual Cell Type Annotation

The process of labeling groups of cells in your data based on known (or sometimes unknown) cellular phenotypes is called “cell annotation”. Whereas there are many ways to annotate your cells (e.g. based on batch, disease, sex and more), in this notebook we will focus on the annotation of “cell types”.

A cell type is a cellular phenotype that is robust across datasets, identifiable by specific marker genes or proteins, and tied to a distinct biological function. A classic example is the plasma B cell — a white blood cell that secretes antibodies and can be identified by characteristic markers.

Previously we identified clusters within our cells. Now we want to label the biological entity that each cluster most likely belongs to. We can use reference **marker gene** lists to do this: previous research has identified genes that are significantly highly or uniquely expressed in one cell type compared to others. We can use these known marker gene expression values in our own dataset to identify what cells belong to what cell types. Below is a marker gene list pulled from existing research: 

In [ ]:
# Azimuth Human PBMC - celltype.l2 (default) marker genes
# Source: https://azimuth.hubmapconsortium.org/references/#Human%20-%20PBMC
# Reference: Hao and Hao et al, Cell 2021

PBMC_CELLTYPE_L2_MARKERS = {
# B cells
 'B intermediate':  ['MS4A1', 'IGHM', 'IGHD', 'CD79A', 'BANK1', 'CD79B'],
 'B memory':        ['MS4A1', 'BANK1', 'CD79A', 'TEX9', 'TNFRSF13C', 'LINC01781'],
 'B naive':         ['IGHM', 'IGHD', 'CD79A', 'MS4A1', 'TCL1A', 'CD79B'],
 'Plasmablast':     ['IGHA2', 'MZB1', 'TNFRSF17', 'DERL3', 'CPNE5', 'NT5DC2'],
# CD4 T cells
 'CD4 CTL':         ['GZMH', 'FGFBP2', 'GNLY', 'B2M', 'NKG7'],
 'CD4 Naive':       ['IL7R'],
 'CD4 Proliferating': ['MKI67', 'TOP2A', 'PCLAF', 'CENPF', 'TYMS', 'NUSAP1', 'ASPM', 'RRM2'],
 'CD4 TCM':         ['IL7R', 'AQP3'],
 'CD4 TEM':         ['IL7R', 'CCL5', 'GZMK', 'KLRB1', 'AQP3'],
 'Treg':            ['IL2RA', 'LAIR2'],
# CD8 T cells
 'CD8 Naive':       ['CD8B', 'S100B', 'RGS10', 'CD8A'],
 'CD8 Proliferating': ['MKI67', 'CD8B', 'TYMS', 'PCLAF', 'CLSPN', 'TK1', 'RRM2'],
 'CD8 TCM':         ['CD8B', 'CD8A', 'IL7R'],
 'CD8 TEM':         ['CCL5', 'GZMH', 'CD8A', 'NKG7', 'GZMK', 'CD8B', 'TRGC2'],
# Dendritic cells
 'ASDC':            ['PPP1R14A', 'AXL', 'LGMN'],
 'cDC1':            ['FLT3'],
 'cDC2':            ['FCER1A', 'HLA-DQA1', 'CLEC10A', 'GSN'],
 'pDC':             ['ITM2C', 'SERPINF1', 'TPM2', 'MZB1'],
# Monocytes
 'CD14 Mono':       ['S100A9','CTSS','S100A8', 'LYZ', 'VCAN', 'S100A12', 'IL1B', 'CD14', 'G0S2', 'FCN1'],
 'CD16 Mono':       ['CDKN1C', 'FCGR3A', 'LST1', 'IFITM3', 'AIF1', 'HES4'],
# NK cells
 'NK':              ['GNLY', 'NKG7', 'FCER1G', 'TRDC', 'PRF1', 'FGFBP2', 'SPON2'],
 'NK Proliferating': ['MKI67', 'TYMS', 'TRDC', 'TOP2A', 'FCER1G', 'PCLAF', 'CLSPN', 'ASPM'],
 'NK_CD56bright':   ['XCL2', 'FCER1G', 'TRDC', 'KLRC1', 'XCL1'],
# Other lineages
 'Eryth':           ['HBD', 'HBM', 'AHSP', 'ALAS2', 'CA1', 'SLC4A1', 'TRIM58', 'SELENBP1', 'TMCC2'],
 'HSPC':            ['PRSS57', 'CYTL1', 'EGFL7', 'GATA2', 'LAPTM4B'],
 'ILC':             ['TRDC', 'SOX4', 'KLRB1'],
 'Platelet':        ['PPBP', 'PF4', 'NRGN', 'GNG11', 'CAVIN2', 'TUBB1', 'CLU', 'HIST1H2AC', 'RGS18', 'GP9'],
 'dnT':             ['CAV1', 'GZMK'],
 'gdT':             ['TRDC', 'TRGC1', 'TRGC2', 'KLRC1', 'NKG7'],
 'MAIT':            ['KLRB1', 'NKG7', 'GZMK', 'IL7R']}

We can visualize these marker gene expression levels in our dataset by subsetting to a cell type of interest and coloring by this group of marker genes on the UMAP: 

In [ ]:
# as an example, we can visualize the expression of NK cell markers on the UMAP 
# we can select all three NK cell subtypes (or just one of them, depending on our interest)
cell_types_of_interest = [
    "NK"
]

# plotting function: 
for ct in cell_types_of_interest:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adata,
        color=PBMC_CELLTYPE_L2_MARKERS[ct], # color the UMAP by the expression of the marker genes for that cell type
        vmin=0,
        vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  
    )

## Exercise: try manual cell type annotations

Cool! It seems we can already see the marker genes naturally being highly expressed in specific areas, or clusters, of our UMAP. 

For simplicity, reference the Leiden clustering at resolution = 0.25 for this. 

Can you identify which Leiden cluster we could assign the NK cell type to? *(Remember that this may not be a perfect fit, but the best one. Even markers for a single cell type are often expressed in different subsets of the data, i.e. individual markers are often not uniquely expressed in a single cell type. Rather, it is the intersection of those subsets that will tell you where your cell type of interest is.)*

Next, try to plot the marker gene expression for a different cell type, e.g. "B intermediate" or "Platelet" by modifying the value in `cell_types_of_interest` in the code cell above and rerunning. Can you do the same thing to annotate which Leiden cluster you think could correspond to these other cell types? 

What about the cell type "CD4 TEM" (Effector Memory CD4+ T cells)? Does this look as clear for manual cell type annotation as the previous cell types? What can this tell you about a manual cell type annotation approach, will this work in 100% of cases? 

### Your manual cell type notes here: 

Cluster numbers refer to Leiden resolution 0.25 here for simplicity! 

NK cells = 

B intermediate cells = 

Platelet cells = 

CD4 TEM cells = 

(Explore more than just these if you'd like! )

## Cell type annotations: identifying differentially expressed genes in each Leiden cluster

Previously, we used existing prior biological knowledge to plot known marker genes' expression in our data. Conversely, we can calculate marker genes per cluster (genes differentially expressed in each cluster) and then look up whether we can link those marker genes to any known biology, such as cell types and/or states. 

Let's find and plot these genes first: 

In [ ]:
res = "leiden_res0_25"

# scanpy function to perform differential expression analysis between the clusters defined by the Leiden clustering at resolution 0.5
# using the statistical Wilcoxon rank-sum test 
sc.tl.rank_genes_groups(
    adata, groupby=res, method="wilcoxon", key_added="dea_leiden_1"
)

# visualize
sc.tl.dendrogram(
    adata,
    groupby=res,
)

sc.pl.rank_genes_groups_dotplot(
    adata, groupby=res, standard_scale="var", n_genes=4, key="dea_leiden_1"
)

## Exercise: interpret the identified 'marker genes' in our data

Have a look at the four most differentially expressed genes identified in **cluster 7**. You can look up each gene and associated cell types by searching, for example, "LST1 cell type". After looking through these four genes identified, what do you think is a probable cell type label for cluster 7? Are you entirely sure that this is correct, or could it be a different, but similar cell type instead? 

Indeed, marker-based annotation can be sensitive to the cluster resolution you choose, the robustness and uniqueness of the marker sets you have, and your knowledge of the cell types to be expected in your data. Therefore, cluster annotations are not guaranteed to be definitive and should be interpreted with caution!

## Best practice: use automated cell type annotations

It is worth noting that the methods discussed so far use only a small subset of the genes detected in the data: often a set of only 1 to ~10 marker genes per cell type is used. An alternative approach is to use a classifier that takes as input a larger set of genes (several thousands or more), thereby making more use of the breadth of scRNA-seq data. Such classifiers are trained on previously annotated datasets or atlases. Examples of these are CellTypist ([Conde *et al.*, 2022](https://www.science.org/doi/full/10.1126/science.abl5197)) which we will use here. 

In [ ]:
adata_celltypist = adata.copy()  # make a copy of our adata
adata_celltypist.X = adata.layers["counts"]  # set adata.X to raw counts
sc.pp.normalize_total(
    adata_celltypist, target_sum=10**4
)  # normalize to 10,000 counts per cell
sc.pp.log1p(adata_celltypist)  # log-transform
# make .X dense instead of sparse, for compatibility with celltypist:
adata_celltypist.X = adata_celltypist.X.toarray()

In [ ]:
# We’ll now download the celltypist models for immune cells:
models.download_models(
    force_update=True, model=["Immune_All_Low.pkl", "Immune_All_High.pkl"]
)

In [ ]:
# let's try out Immune_All_Low first: finer annotation level i.e. more fine-grained cell types: 
model_low = models.Model.load(model="Immune_All_Low.pkl")
# we can also try out Immune_All_High, which is a coarser model with fewer, more general cell types:
model_high = models.Model.load(model="Immune_All_High.pkl")
# let's look at the cell types in the high model:
model_high.cell_types

In [ ]:
# run coarse annotation with the high-hierarchy model: 
predictions_high = celltypist.annotate(
    adata_celltypist, model=model_high, majority_voting=True
)

In [ ]:
# run fine annotation with the low-hierarchy model:
predictions_low = celltypist.annotate(
    adata_celltypist, model=model_low, majority_voting=True
)

In [ ]:
# Transform the predictions to adata to get the full output
# and copy the results to our original AnnData object 
# both for the coarse adn fine annotation levels:

predictions_high_adata = predictions_high.to_adata()
adata.obs["celltypist_cell_label_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index, "majority_voting"
]
adata.obs["celltypist_conf_score_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index, "conf_score"
]

predictions_low_adata = predictions_low.to_adata()
adata.obs["celltypist_cell_label_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index, "majority_voting"
]
adata.obs["celltypist_conf_score_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index, "conf_score"
]

In [ ]:
# now we can visualize the celltypist annotations on the UMAP, along with the confidence scores for each annotation level:

sc.pl.umap(
    adata,
    color=["celltypist_cell_label_coarse", "celltypist_conf_score_coarse", "celltypist_cell_label_fine", "celltypist_conf_score_fine"],
    frameon=False,
    sort_order=False,
    wspace=0.7,
    ncols=2
)

## Exercise: 

Take some time to familiarize yourself with the cell type annotations generated by CellTypist above. Are you familiar with all the coarse cell types? How does the clustering compare to the Leiden algorithm earlier? Make sure you are confident in the big picture: why do we annotate clusters of cells by their cell identify, and how this is helpful for downstream analysis? (When looking into an inflamatory immune response, do we expect to see the same reaction in e.g. B Cells and Monocytes?)

## Optional extension exercise: 
### Only focus on this if you have more than one hour left, otherwise move on to the following section! 

Based on previous experiences with manual cell type annotations and the CellTypist confidence score, do you think you should accept these annotations without further review? 

One way of getting a feeling for the quality of these annotations is by looking if the observed cell type similarities correspond to our expectations: run `sc.tl.dendogram` with `groupby = "celltypist_cell_label_fine"` followed by `sc.pl.dendogram` with the same parameters to visualize the hierarchical clustering of the identified fine cell types. Are B cells largely clustered together as well expect? Is the same true for T cells? 

It is very important to check the automated annotation manually before simply accepting them! For the purposes of this tutorial, the given CellTypist annotations are fine for now. 

## Downstream Analysis: differential gene expression of COVID-19 vs healthy patients

Previously, we used genes expressed differentially in identified clusters to inspect cell type annotations. Here, we focus on more advanced use-cases of differential gene expression testing on more complex experimental designs which involve one or more conditions such as diseases, genetic knockouts or drugs. In such cases we are commonly interested in the magnitude and significance of differences in gene expression patterns between the condition of interest and a reference. This reference can be everything but is commonly a healthy sample. This statistical test can be applied to arbitrary groups, but in the case of single-cell RNA-Seq is commonly applied on the cell type level.

<img src=https://www.sc-best-practices.org/_images/differential_gene_expression.jpg width=900>

A differential gene expression test usually returns the **log2 fold-change** and the **adjusted p-value** per compared genes per compared conditions. This list can then be sorted by p-value and investigated in more detail.

#### What is log2 fold-change? 

$$ log_2 FC = log_2 ( \frac{COVID19}{Healthy} )$$
where $COVID19$ and $Healthy$ values are the average gene expression in a pseudobulk patient's specific cell type. We can test, for each gene of interest, what the difference in these expression values are and thus identify genes where their expression changes significantly across disease states. Using the logarithm allows us to achieve more interpretable results: it centers the values around $0$ (no change in expression between the two groups) and is symmetrical around zero, so $-2$ corresponds to a halving in gene expression while $2$ corresponds to a doubling of gene expression. 

#### What are the adjusted p-values? 

If you run one statistical test with a standard significance level ($\alpha = 0.05$), you have a $95\%$ chance of getting *not* getting a false positive. If you run 100 independent tests, the probability of getting at least one false positive skyrockets: 
For 100 independent tests, the probability of getting *no* false positives at all is
$ 0.95^{100}≈0.0059≈0.5\%$, that is, we are most likely to get many false positives and need to correct for this. 

Adjusted p-values mathematically scale our results to compensate for this, as we are testing at least 2,000 genes in each patient! 
Here, we use a false discovery rate of 0.01 to correct for this: FDR controls the *proportion* of false positives among the discoveries, so we're saying across the whole list of hits, we expect 1% to be noise, which is fine with these types of high-throughput experiments. 

### Pseudobulk for PyDESeq2

Since DESeq2 was introduced as a method for differential expression (DE) analysis for bulk data (RNA sequencing of an entire mixed tissue instead of single cells), we first need to create **pseudobulk** samples from our single-cell dataset. For each patient we create 1 pseudobulk sample per cell type by aggregating the cell from each subpopulation and taking the mean gene expression within that subpopulation.

Then, a popular task in the analysis of count data from RNA-seq is the detection of differentially expressed genes. The count data are presented as a table which reports, for each sample, the number of sequence fragments that have been assigned to each gene. An important analysis question is the quantification and statistical inference of systematic changes between conditions, as compared to within-condition variability. The package DESeq2 provides **methods to test for differential expression** by use of negative binomial generalized linear models; the estimates of dispersion and logarithmic fold changes incorporate data-driven prior distributions. ([Love *et al.*, 2014](https://pmc.ncbi.nlm.nih.gov/articles/PMC4302049/))

In [ ]:
# first we will construct the pseudobulk count matrix
# don't worry about the details too much here as this is a means to an end!

MIN_CELLS = 30  # same threshold as aggregate_and_filter()

cell_type_deseq = "Monocytes"
adata_sub = adata[adata.obs["celltypist_cell_label_coarse"] == cell_type_deseq].copy()

# ── Filter donors with too few cells ─────────────────────────────────────────
cells_per_donor = adata_sub.obs.groupby("sample_id").size()
donors_to_keep  = cells_per_donor[cells_per_donor > MIN_CELLS].index
n_dropped = (cells_per_donor <= MIN_CELLS).sum()
print(f"Dropping {n_dropped} donors with ≤ {MIN_CELLS} cells")

adata_sub = adata_sub[adata_sub.obs["sample_id"].isin(donors_to_keep)].copy()

# ── Build pseudobulk metadata ─────────────────────────────────────────────────
pseudobulk_meta = (
    adata_sub.obs
    .groupby("sample_id")[["disease_state"]]
    .first()
)

# ── Build count matrix ────────────────────────────────────────────────────────
counts_pb = pd.DataFrame(
    index=pseudobulk_meta.index,
    columns=adata_sub.var_names,
    dtype=int
)
for donor in pseudobulk_meta.index:
    mask = adata_sub.obs["sample_id"] == donor
    X = adata_sub[mask].layers["counts"]
    counts_pb.loc[donor] = (
        np.asarray(X.sum(axis=0)).flatten().astype(int)
        if sp.issparse(X)
        else np.asarray(X).sum(axis=0).flatten().astype(int)
    )

print(f"Pseudobulk matrix: {counts_pb.shape}")  # (n_donors, n_genes)

In [ ]:
# Now we Fit ONE shared DESeq2 model across all the disease states we have in the dataset
# we also specify the reference level for the disease_state variable

reference_level = "Healthy donor"  # set reference level for disease_state variable

dds = DeseqDataSet(
    counts=counts_pb,
    metadata=pseudobulk_meta,
    design_factors="disease_state",
    ref_level=["disease_state", reference_level]  # reference level for all contrasts
)
dds.deseq2()

In [ ]:
# Now we can extract three contrasts of interest against Healthy 
contrasts = {
    "Asymptomatic_vs_Reference": ["disease_state", "mild COVID-19 (asymptomatic)", reference_level],
    "Mild_vs_Reference":         ["disease_state", "mild COVID-19",         reference_level],
    "Severe_vs_Reference":       ["disease_state", "severe COVID-19",       reference_level],
}

results = {}
# here we loop through the contrasts, extract the DESeq2 results for each contrast, and store them for plotting later
for name, contrast in contrasts.items():
    stat_res = DeseqStats(dds, contrast=contrast)
    stat_res.summary()
    df = stat_res.results_df.copy()
    df["contrast"] = name
    results[name] = df

In [ ]:
# this code defines two functions: volcano_plot_pydeseq2() to make a volcano plot for a single contrast, 
# and volcano_plot_multi_contrast() to make side-by-side volcano plots for multiple contrasts. 
# We will use these functions to visualize our DESeq2 results.
# Don't worry about the details too much here as this is a means to an end!

FDR = 0.01
LOG_FOLD_CHANGE = 1.5

def volcano_plot_pydeseq2(
    # Volcano plot for PyDESeq2 results DataFrame.
    results_df,
    lfc_col="log2FoldChange",
    pval_col="padj",
    fdr_threshold=FDR,
    lfc_threshold=LOG_FOLD_CHANGE,
    title=None,
    label_top_n=10,           # label the top N most significant genes
    ax=None
):

    # Clean & compute -log10(padj) 
    result = results_df.dropna(subset=[lfc_col, pval_col]).copy()
    result["-logQ"] = -np.log10(result[pval_col].astype(float))

    # Split into significance categories 
    sig_both = result.loc[
        (result[pval_col] < fdr_threshold) &
        (result[lfc_col].abs() > lfc_threshold)
    ]
    sig_pval_only = result.loc[
        (result[pval_col] < fdr_threshold) &
        (result[lfc_col].abs() <= lfc_threshold)
    ]
    sig_lfc_only = result.loc[
        (result[pval_col] >= fdr_threshold) &
        (result[lfc_col].abs() > lfc_threshold)
    ]
    not_sig = result.loc[
        (result[pval_col] >= fdr_threshold) &
        (result[lfc_col].abs() <= lfc_threshold)
    ]

    # Plot 
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Color scheme: grey, blue, orange, red
    for subset, color, label in [
        (not_sig,       "lightgrey", "Not significant"),
        (sig_pval_only, "steelblue", f"FDR < {fdr_threshold}"),
        (sig_lfc_only,  "orange",    f"|log2FC| > {lfc_threshold}"),
        (sig_both,      "crimson",   f"FDR < {fdr_threshold} & |log2FC| > {lfc_threshold}"),
    ]:
        sns.regplot(
            x=subset[lfc_col],
            y=subset["-logQ"],
            fit_reg=False,
            scatter_kws={"s": 6, "alpha": 0.7, "color": color},
            label=label,
            ax=ax
        )

    # Threshold lines 
    ax.axhline(
        y=-np.log10(fdr_threshold),
        linestyle="--", color="black",
        linewidth=0.8, alpha=0.6
    )
    ax.axvline(
        x=lfc_threshold,
        linestyle="--", color="black",
        linewidth=0.8, alpha=0.6
    )
    ax.axvline(
        x=-lfc_threshold,
        linestyle="--", color="black",
        linewidth=0.8, alpha=0.6
    )

    # Label top N significant genes 
    if label_top_n > 0 and len(sig_both) > 0:
        top_genes = sig_both.nlargest(label_top_n, "-logQ")
        for gene, row in top_genes.iterrows():
            ax.text(
                row[lfc_col],
                row["-logQ"],
                gene,
                fontsize=7,
                ha="left",
                va="bottom"
            )

    # Labels & formatting 
    ax.set_xlabel("log2 Fold Change", fontsize=11)
    ax.set_ylabel("-log10(adjusted p-value)", fontsize=11)
    ax.set_title(title or "Volcano Plot", fontsize=13, fontweight="bold")
    ax.legend(markerscale=2, fontsize=8, loc="upper left")

    plt.tight_layout()
    return ax

def volcano_plot_multi_contrast(
    # Side-by-side volcano plots for multiple PyDESeq2 contrasts.
    results_dict,
    lfc_threshold=LOG_FOLD_CHANGE,
    fdr_threshold=FDR,
    cell_type_name="",
    label_top_n=8,
    figsize=(16, 5)
):

    n = len(results_dict)
    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=True)

    # Make axes always iterable even if n=1
    if n == 1:
        axes = [axes]

    for ax, (contrast_name, results_df) in zip(axes, results_dict.items()):
        # Clean contrast name for title: "Severe_vs_Reference" -> "Severe vs Reference"
        clean_title = contrast_name.replace("_", " ")

        volcano_plot_pydeseq2(
            results_df,
            lfc_threshold=lfc_threshold,
            fdr_threshold=fdr_threshold,
            title=clean_title,
            label_top_n=label_top_n,
            ax=ax
        )

        # Only show legend on last subplot to avoid repetition
        if ax != axes[0]:
            ax.get_legend().remove()

    fig.suptitle(
        f"COVID-19 Severity vs {reference_level}  |  {cell_type_name}",
        fontsize=14,
        fontweight="bold",
        y=1.02
    )
    plt.tight_layout()
    plt.show()

In [ ]:
volcano_plot_multi_contrast(
    results_dict=results,   
    cell_type_name=cell_type_deseq,
    lfc_threshold=1.5,
    fdr_threshold=0.01,
    label_top_n=8
)

## Exercise: find differentially expressed genes between mild and severe COVID-19

You've now ran the DESeq2 analysis using the value `Healthy donor` as the reference. However, we could also be interested in what makes severe COVID-19 different in gene expression in monocytes compared to mild COVID-19. Go through the code above and run it again, but change the reference to the value `mild COVID-19` instead. 

When you get the results, look up some of the most differentially expressed genes. What biological pathways do they appear in? Is there existing research about their impact in COVID-19 or other viral infections? 

This is where you take the role of a researcher: your statistically robust findings can lead to further lines of inquiry and importantly identify real biological differences between disease states! 

## Extension

There are many, many more possible downstream analysis we could do with our existing scRNASeq data. Have a look at the [best practices](https://www.sc-best-practices.org/preamble.html) or the [paper](https://www.science.org/doi/10.1126/sciimmunol.abd1554) we pulled this data from if you're interested in learning more about these. 